# PCR Monte Carlo simulations

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


# **PCA NON-LINEAR total R^2**

In [ ]:
!pip install tables
!pip install ipca

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
import os

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_NON_LINEAR_TOTAL')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Check if the directory was created successfully
if results_path.exists():
    print(f"Directory successfully created: {results_path}")
else:
    print(f"Failed to create directory: {results_path}")

# Parameters
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']

# Function to generate data
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([
            np.power(1.2 * c_normalized[:, 0, t], 3),
            c_normalized[:, 1, t] * c_normalized[:, 0, t],
            np.power(0.4 * c_normalized[:, 2, t], 2),
            2 * c_normalized[:, 3, t] * c_normalized[:, 2, t],
            np.power(0.6 * c_normalized[:, 4, t], 4)
        ]).T
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1
    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]
    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

# Helper functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

# Loop over each factor and calculate average R^2 for all simulations
scores_dict = {}
for n_factors in tqdm(factors):
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
        y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
        y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)

        # Initialize PCA and Linear Regression model
        pca = PCA(n_components=n_factors)
        regr = LinearRegression()

        # Fit PCA and transform training data
        X_train_pca = pca.fit_transform(X_train)
        regr.fit(X_train_pca, y_train)

        # Transform test data and make predictions
        X_test_pca = pca.transform(X_test)
        y_pred = regr.predict(X_test_pca)
        y_pred = np.array(y_pred).flatten()

        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
        r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])
        factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)
        sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)
        y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))
        factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)
        r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
        portfolio_returns = []
        for t in range(r_t_test.shape[1]):
            actual_returns = r_t_test[:, t]
            predicted_returns = y_pred_reshaped[:, t]
            if predicted_returns.shape == actual_returns.shape:
                portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                portfolio_returns.append(portfolio_return)
            else:
                print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")
        portfolio_returns = np.array(portfolio_returns)
        long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)
        prediction_df = pd.DataFrame({
            'predicted returns': y_pred,
            'actual returns': y_test
        })
        print(f'| FACTOR : {n_factors} | SIMULATION: {i} | TEST R^2, r_t: {r2} |TEST R^2, x_t : {r2_x_t} ')
        print(f"Simulation {i+1}, Factors {n_factors} - R-squared (R²): {r2}")
        scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {hdf5_path}")
    try:
        scores_df.to_hdf(str(hdf5_path), key='scores', mode='w')
        print(f"Scores successfully saved to {hdf5_path}")
    except Exception as e:
        print(f"Error saving scores to {hdf5_path}: {e}")


**LOAD DATA**

In [ ]:
import pandas as pd
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_NON_LINEAR_TOTAL')
results_path = base_path / 'results' / 'asset_pricing'

# Initialize an empty list to store the DataFrames
all_scores = []

# Loop over the factors to load each HDF5 file
factors = [1, 2, 3, 4, 5, 6]
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    if hdf5_path.exists():
        try:
            scores_df = pd.read_hdf(hdf5_path, key='scores')
            scores_df['Factors'] = n_factors  # Add a column to identify the number of factors
            all_scores.append(scores_df)
            print(f"Successfully loaded scores from {hdf5_path}")
        except Exception as e:
            print(f"Error loading scores from {hdf5_path}: {e}")
    else:
        print(f"File does not exist: {hdf5_path}")

# Concatenate all DataFrames into a single DataFrame
if all_scores:
    combined_scores_df = pd.concat(all_scores, ignore_index=True)
    print("All scores loaded successfully. Here are the combined results:")
    display(combined_scores_df)  # Use display() to show the DataFrame in Google Colab
else:
    print("No scores were loaded.")

# Optionally, save the combined results to a new HDF5 file
combined_hdf5_path = results_path / 'combined_scores.h5'
try:
    combined_scores_df.to_hdf(combined_hdf5_path, key='combined_scores', mode='w')
    print(f"Combined scores successfully saved to {combined_hdf5_path}")
except Exception as e:
    print(f"Error saving combined scores to {combined_hdf5_path}: {e}")


In [ ]:
import pandas as pd
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_NON_LINEAR_TOTAL')
results_path = base_path / 'results' / 'asset_pricing'

# Initialize an empty list to store the DataFrames
all_scores = []

# Loop over the factors to load each HDF5 file
factors = [1, 2, 3, 4, 5, 6]
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    if hdf5_path.exists():
        try:
            scores_df = pd.read_hdf(hdf5_path, key='scores')
            scores_df['Factors'] = n_factors  # Add a column to identify the number of factors
            all_scores.append(scores_df)
            print(f"Successfully loaded scores from {hdf5_path}")
        except Exception as e:
            print(f"Error loading scores from {hdf5_path}: {e}")
    else:
        print(f"File does not exist: {hdf5_path}")

# Concatenate all DataFrames into a single DataFrame
if all_scores:
    combined_scores_df = pd.concat(all_scores, ignore_index=True)
    print("All scores loaded successfully. Here are the combined results:")
    display(combined_scores_df)  # Use display() to show the DataFrame in Google Colab
else:
    print("No scores were loaded.")

# Optionally, save the combined results to a new HDF5 file
combined_hdf5_path = results_path / 'combined_scores.h5'
try:
    combined_scores_df.to_hdf(combined_hdf5_path, key='combined_scores', mode='w')
    print(f"Combined scores successfully saved to {combined_hdf5_path}")
except Exception as e:
    print(f"Error saving combined scores to {combined_hdf5_path}: {e}")


#**PCA LINEAR, TOTAL $R^2$**

In [ ]:





!pip install tables
!pip install ipca

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from pathlib import Path
from tqdm import tqdm
from google.colab import drive
import os

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_LINEAR_TOTAL')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Check if the directory was created successfully
if results_path.exists():
    print(f"Directory successfully created: {results_path}")
else:
    print(f"Failed to create directory: {results_path}")

# Parameters
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]

cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']

# Function to generate data
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1
    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]
    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

# Helper functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

# Loop over each factor and calculate average R^2 for all simulations
scores_dict = {}
for n_factors in tqdm(factors):
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size + 1)], axis=0)
        y_train = np.concatenate([train_data[:, t] for t in range(train_size)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T)], axis=0)
        y_test = np.concatenate([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)], axis=0)

        # Initialize PCA and Linear Regression model
        pca = PCA(n_components=n_factors)
        regr = LinearRegression()

        # Fit PCA and transform training data
        X_train_pca = pca.fit_transform(X_train)
        regr.fit(X_train_pca, y_train)

        # Transform test data and make predictions
        X_test_pca = pca.transform(X_test)
        y_pred = regr.predict(X_test_pca)
        y_pred = np.array(y_pred).flatten()

        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
        r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])
        factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)
        sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)
        y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))
        factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)
        r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
        portfolio_returns = []
        for t in range(r_t_test.shape[1]):
            actual_returns = r_t_test[:, t]
            predicted_returns = y_pred_reshaped[:, t]
            if predicted_returns.shape == actual_returns.shape:
                portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                portfolio_returns.append(portfolio_return)
            else:
                print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")
        portfolio_returns = np.array(portfolio_returns)
        long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)
        prediction_df = pd.DataFrame({
            'predicted returns': y_pred,
            'actual returns': y_test
        })
        print(f'| FACTOR : {n_factors} | SIMULATION: {i} | TEST R^2, r_t: {r2} |TEST R^2, x_t : {r2_x_t} ')
        print(f"Simulation {i+1}, Factors {n_factors} - R-squared (R²): {r2}")
        scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {hdf5_path}")
    try:
        scores_df.to_hdf(str(hdf5_path), key='scores', mode='w')
        print(f"Scores successfully saved to {hdf5_path}")
    except Exception as e:
        print(f"Error saving scores to {hdf5_path}: {e}")


# **PCA LINEAR PREDICTIVE $R^2$**

In [ ]:
!pip install tables


import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_LINEAR_PREDICTIVE')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Parameters
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]
cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']

# Function to generate data
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1
    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]
    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

# Helper functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

def calculate_forward_returns(returns, forward_period=1):
    return returns[:, forward_period:]

# Main loop for simulations
scores_dict = {}
for n_factors in tqdm(factors):
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        # Calculate forward returns
        train_forward_returns = calculate_forward_returns(train_data, forward_period=1)
        val_forward_returns = calculate_forward_returns(val_data, forward_period=1)
        test_forward_returns = calculate_forward_returns(test_data, forward_period=1)

        # Adjust the time range to match forward returns
        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size)], axis=0)
        y_train = np.concatenate([train_forward_returns[:, t] for t in range(train_size - 1)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T - 1)], axis=0)
        y_test = np.concatenate([test_forward_returns[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)], axis=0)

        # Initialize PCA and Linear Regression model
        pca = PCA(n_components=n_factors)
        regr = LinearRegression()

        # Fit PCA and transform training data
        X_train_pca = pca.fit_transform(X_train)
        regr.fit(X_train_pca, y_train)

        # Transform test data and make predictions
        X_test_pca = pca.transform(X_test)
        y_pred = regr.predict(X_test_pca)
        y_pred = np.array(y_pred).flatten()

        # Calculate R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Adjust shapes for factor network input calculation
        Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T - 1)])
        r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)])

        factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)
        sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)

        # Reshape y_pred to match r_t_test shape
        y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))

        factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)

        r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
        portfolio_returns = []
        for t in range(r_t_test.shape[1]):
            actual_returns = r_t_test[:, t]
            predicted_returns = y_pred_reshaped[:, t]
            if predicted_returns.shape == actual_returns.shape:
                portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                portfolio_returns.append(portfolio_return)
            else:
                print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")
        portfolio_returns = np.array(portfolio_returns)
        long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

        # Save scores
        scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {hdf5_path}")
    try:
        scores_df.to_hdf(str(hdf5_path), key='scores', mode='w')
        print(f"Scores successfully saved to {hdf5_path}")
    except Exception as e:
        print(f"Error saving scores to {hdf5_path}: {e}")

#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors + 1):
        file_path = results_path / f'scores_{n_factors}.h5'
        if file_path.exists():
            try:
                scores_df = pd.read_hdf(file_path, key='scores')
                scores_dict[n_factors] = scores_df
                print(f"Successfully loaded scores from {file_path}")
            except Exception as e:
                print(f"Error loading scores from {file_path}: {e}")
        else:
            print(f"File does not exist: {file_path}")
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)
if scores_dict_loaded:
    combined_scores_df = pd.concat(scores_dict_loaded.values(), ignore_index=True)
    print("All scores loaded successfully. Here are the combined results:")
    display(combined_scores_df)

    # Optionally, save the combined results to a new HDF5 file
    combined_hdf5_path = results_path / 'combined_scores.h5'
    try:
        combined_scores_df.to_hdf(combined_hdf5_path, key='combined_scores', mode='w')
        print(f"Combined scores successfully saved to {combined_hdf5_path}")
    except Exception as e:
        print(f"Error saving combined scores to {combined_hdf5_path}: {e}")
else:
    print("No scores were loaded.")


# **PCA NON-LINEAR, PREDICTIVE $R^2$**

In [ ]:

!pip install tables


import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from pathlib import Path
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_NON_LINEAR_PREDICTIVE')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    print(f"Creating directory: {results_path}")
    results_path.mkdir(parents=True)
else:
    print(f"Directory already exists: {results_path}")

# Parameters
N = 200
T = 180
Pc = 50
Px = 50
num_factors = 5
R2_target = 0.45
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]
cols = ['Factors', 'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio', 'R^2 for r_t', 'Adjusted R^2', 'R^2 for x_t']

# Function to generate data
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([
            np.power(1.2 * c_normalized[:, 0, t], 3),
            c_normalized[:, 1, t] * c_normalized[:, 0, t],
            np.power(0.4 * c_normalized[:, 2, t], 2),
            2 * c_normalized[:, 3, t] * c_normalized[:, 2, t],
            np.power(0.6 * c_normalized[:, 4, t], 4)
        ]).T

    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1
    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size + val_size]
    test_data = r_adjusted[:, train_size + val_size:]
    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size

# Helper functions
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[2]):
        Z_t = Z_t_minus_1[:, :, t]
        r_t_i = r_t[:, t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    N = predicted_returns.shape[0]
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold
    long_returns = actual_returns[long_positions].mean()
    short_returns = actual_returns[short_positions].mean()
    portfolio_return = long_returns - short_returns
    return portfolio_return

def calculate_forward_returns(returns, forward_period=1):
    return returns[:, forward_period:]

# Main loop for simulations
scores_dict = {}
for n_factors in tqdm(factors):
    r2_scores = []
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        # Calculate forward returns
        train_forward_returns = calculate_forward_returns(train_data, forward_period=1)
        val_forward_returns = calculate_forward_returns(val_data, forward_period=1)
        test_forward_returns = calculate_forward_returns(test_data, forward_period=1)

        # Adjust the time range to match forward returns
        X_train = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(1, train_size)], axis=0)
        y_train = np.concatenate([train_forward_returns[:, t] for t in range(train_size - 1)], axis=0)
        X_test = np.concatenate([c_normalized[:, :, t].reshape(N, Pc) for t in range(train_size + val_size + 1, T - 1)], axis=0)
        y_test = np.concatenate([test_forward_returns[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)], axis=0)

        # Initialize PCA and Linear Regression model
        pca = PCA(n_components=n_factors)
        regr = LinearRegression()

        # Fit PCA and transform training data
        X_train_pca = pca.fit_transform(X_train)
        regr.fit(X_train_pca, y_train)

        # Transform test data and make predictions
        X_test_pca = pca.transform(X_test)
        y_pred = regr.predict(X_test_pca)
        y_pred = np.array(y_pred).flatten()

        # Calculate R^2 score
        r2 = r2_score(y_test, y_pred)
        r2_scores.append(r2)

        # Adjust shapes for factor network input calculation
        Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T - 1)])
        r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T - 1)])

        factor_network_input_test = calculate_factor_network_input(Z_t_minus_1_test, r_t_test)
        sharpe_ratio_x_t = calculate_sharpe_ratio(factor_network_input_test)

        # Reshape y_pred to match r_t_test shape
        y_pred_reshaped = y_pred.reshape((r_t_test.shape[0], r_t_test.shape[1]))

        factor_network_input_pred = calculate_factor_network_input(Z_t_minus_1_test, y_pred_reshaped)

        r2_x_t = r2_score(factor_network_input_test, factor_network_input_pred)
        portfolio_returns = []
        for t in range(r_t_test.shape[1]):
            actual_returns = r_t_test[:, t]
            predicted_returns = y_pred_reshaped[:, t]
            if predicted_returns.shape == actual_returns.shape:
                portfolio_return = calculate_long_short_portfolio(predicted_returns, actual_returns)
                portfolio_returns.append(portfolio_return)
            else:
                print(f"Shape mismatch: predicted_returns.shape = {predicted_returns.shape}, actual_returns.shape = {actual_returns.shape}")
        portfolio_returns = np.array(portfolio_returns)
        long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

        # Save scores
        scores.append([n_factors, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, r2, adjusted_R2, r2_x_t])

    # Save scores to HDF5
    scores_df = pd.DataFrame(scores, columns=cols)
    scores_dict[n_factors] = scores_df
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    print(f"Saving scores to {hdf5_path}")
    try:
        scores_df.to_hdf(str(hdf5_path), key='scores', mode='w')
        print(f"Scores successfully saved to {hdf5_path}")
    except Exception as e:
        print(f"Error saving scores to {hdf5_path}: {e}")

#############################
## FUNCTION TO LOAD SCORES ##
#############################

def load_scores(results_path, num_factors):
    scores_dict = {}
    for n_factors in range(1, num_factors + 1):
        file_path = results_path / f'scores_{n_factors}.h5'
        if file_path.exists():
            try:
                scores_df = pd.read_hdf(file_path, key='scores')
                scores_dict[n_factors] = scores_df
                print(f"Successfully loaded scores from {file_path}")
            except Exception as e:
                print(f"Error loading scores from {file_path}: {e}")
        else:
            print(f"File does not exist: {file_path}")
    return scores_dict

# Example usage to load the scores
scores_dict_loaded = load_scores(results_path, num_factors=6)
if scores_dict_loaded:
    combined_scores_df = pd.concat(scores_dict_loaded.values(), ignore_index=True)
    print("All scores loaded successfully. Here are the combined results:")
    display(combined_scores_df)

    # Optionally, save the combined results to a new HDF5 file
    combined_hdf5_path = results_path / 'combined_scores.h5'
    try:
        combined_scores_df.to_hdf(combined_hdf5_path, key='combined_scores', mode='w')
        print(f"Combined scores successfully saved to {combined_hdf5_path}")
    except Exception as e:
        print(f"Error saving combined scores to {combined_hdf5_path}: {e}")
else:
    print("No scores were loaded.")


In [ ]:
import pandas as pd
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install pytables
!pip install tables

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/PCA_SIMULATIONS_LINEAR_TOTAL')
results_path = base_path / 'results' / 'asset_pricing'

# Initialize an empty list to store the DataFrames
all_scores = []

# Loop over the factors to load each HDF5 file
factors = [1, 2, 3, 4, 5, 6]
for n_factors in factors:
    hdf5_path = results_path / f'scores_{n_factors}.h5'
    if hdf5_path.exists():
        try:
            scores_df = pd.read_hdf(hdf5_path, key='scores')
            scores_df['Factors'] = n_factors  # Add a column to identify the number of factors
            all_scores.append(scores_df)
            print(f"Successfully loaded scores from {hdf5_path}")
        except Exception as e:
            print(f"Error loading scores from {hdf5_path}: {e}")
    else:
        print(f"File does not exist: {hdf5_path}")

# Concatenate all DataFrames into a single DataFrame
if all_scores:
    combined_scores_df = pd.concat(all_scores, ignore_index=True)
    print("All scores loaded successfully. Here are the combined results:")
    display(combined_scores_df)  # Use display() to show the DataFrame in Google Colab

    # Calculate the mean of each factor
    mean_scores = combined_scores_df.groupby('Factors').mean()

    # Convert the mean scores DataFrame to a LaTeX table
    latex_table = mean_scores.to_latex()
    print("Mean scores for each factor (in LaTeX format):")
    print(latex_table)
else:
    print("No scores were loaded.")

# Optionally, save the combined results to a new HDF5 file
combined_hdf5_path = results_path / 'combined_scores.h5'
try:
    combined_scores_df.to_hdf(combined_hdf5_path, key='combined_scores', mode='w')
    print(f"Combined scores successfully saved to {combined_hdf5_path}")
except Exception as e:
    print(f"Error saving combined scores to {combined_hdf5_path}: {e}")


In [ ]:
combined_scores_df.mean()